# Phase 11 (Pacific) - PatchTST Training

Run cells top to bottom. Before running, go to **Runtime > Change runtime type > T4 GPU**.

Same workflow as the Atlantic notebook, pointed at Pacific's data instead.

## Step 0: Confirm GPU is active

In [ ]:
!nvidia-smi

## Step 1: Clone the repo

In [ ]:
!git clone https://github.com/nive62tech/patchtst.git
%cd patchtst/wave-forecasting

## Step 2: Install dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

## Step 3: Get the Pacific windowed data onto Colab

The `.npy` files under `pacific/data/windows/` are gitignored, so they are NOT in the repo.

**Before running the next cell:** upload the 6 files from your local `pacific\data\windows\` folder to Google Drive, e.g. `MyDrive/patchtst_data/pacific_windows/` (use a different folder from Atlantic's, to avoid mixing them up).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Adjust this path if you used a different Drive folder name
DRIVE_WINDOWS_PATH = '/content/drive/MyDrive/patchtst_data/pacific_windows'

!mkdir -p pacific/data/windows
!cp {DRIVE_WINDOWS_PATH}/*.npy pacific/data/windows/
!ls -la pacific/data/windows/

## Step 4: Verify data landed correctly

In [ ]:
import numpy as np
for name in ['X_train', 'y_class_train', 'y_forecast_train', 'X_test', 'y_class_test', 'y_forecast_test']:
    arr = np.load(f'pacific/data/windows/{name}.npy')
    print(f'{name}: {arr.shape} {arr.dtype}')

Expected: `X_train (162520, 72, 6)`, `y_class_train (162520,)`, `y_forecast_train (162520, 20, 3)`, `X_test (69584, 72, 6)`, `y_class_test (69584,)`, `y_forecast_test (69584, 20, 3)` - same shapes as Atlantic, since Pacific has identical row counts.

## Step 5: Run training

In [ ]:
import sys
sys.path.insert(0, '.')

from src.train_pacific import train_model

history = train_model(num_epochs=100, device='cuda')

## Step 6: Plot training curves

Saved to `pacific/results/training_curves.png`.

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs('pacific/results', exist_ok=True)

epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_ran, history['train_loss'], label='Train Loss')
axes[0].plot(epochs_ran, history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Joint Loss')
axes[0].set_title('Training vs Validation Loss (Pacific)')
axes[0].legend()

axes[1].plot(epochs_ran, history['val_accuracy'], color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy (Pacific)')

axes[2].plot(epochs_ran, history['val_f1'], color='purple')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Weighted F1')
axes[2].set_title('Validation F1 (Pacific)')

plt.tight_layout()
plt.savefig('pacific/results/training_curves.png', dpi=150)
plt.show()

print('Saved pacific/results/training_curves.png')

## Step 7: Copy outputs to Drive, then download to your local machine

In [ ]:
DRIVE_OUTPUT_PATH = '/content/drive/MyDrive/patchtst_data/pacific_outputs'
!mkdir -p {DRIVE_OUTPUT_PATH}
!cp pacific/pacific_patchtst_best.pt {DRIVE_OUTPUT_PATH}/
!cp pacific/results/training_curves.png {DRIVE_OUTPUT_PATH}/
!ls -la {DRIVE_OUTPUT_PATH}
print('Copied to Drive - download pacific_patchtst_best.pt and training_curves.png from there to your local machine.')

## Step 8: Print final summary (paste this into chat when done)

In [ ]:
print(f"Epochs run: {len(history['train_loss'])}")
print(f"Best val_loss: {min(history['val_loss']):.4f}")
print(f"Final val_accuracy: {history['val_accuracy'][-1]:.4f}")
print(f"Final val_f1: {history['val_f1'][-1]:.4f}")
print(f"Final val_rmse: {history['val_rmse'][-1]:.4f}")